# VENTUREGENESIS — Funding Readiness Model

**Target:** `funding_model.joblib` — `P(startup reaches a fundable / success outcome)`.

A company is **positive** when its `status` is `Acquired` or `Public`, **or** it is flagged
`top_company` in the YC dataset; everything else (including `Inactive`) is negative. This is
the success counterpart to the failure model.

> This notebook **faithfully reproduces the production artifact** that the app loads at
> runtime. The same pipeline lives in `backend/app/ml_training/train.py` (the `funding_model`
> branch) and `backend/app/ml_training/features.py`. Running every cell regenerates
> `models/funding_model.joblib` — equivalent to `python -m app.ml_training.train`.

**Pipeline:** 6 founder-suppliable features → leakage-safe `TargetEncoder` on industry →
`HistGradientBoostingClassifier` (5-fold grid search) → isotonic probability calibration.

## 0 · Setup

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    brier_score_loss, f1_score, accuracy_score, confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RNG = 42

HERE = Path.cwd()
DATA = next(
    p for p in [HERE / "data", HERE.parent / "data"]
    if (p / "yc_companies_algolia.csv").exists()
)
MODELS = (DATA.parent / "models"); MODELS.mkdir(exist_ok=True)
print("data dir :", DATA)
print("models   :", MODELS)

## 1 · Load the pooled YC + Failory dataset

Same two CSVs the trainer concatenates. We keep only rows with a known `status`.

In [ ]:
frames = []
for fname in ["yc_companies_algolia.csv", "failory_dataset_yc_format.csv"]:
    f = DATA / fname
    if f.exists():
        d = pd.read_csv(f); d["source"] = "yc" if "yc" in fname else "failory"
        frames.append(d)
df = pd.concat(frames, ignore_index=True)
df = df[df["status"].notna()].reset_index(drop=True)
print(f"{len(df):,} companies with a known status")
print(df["status"].astype(str).str.strip().str.lower().value_counts())

## 2 · Define the funding / success label

`y = 1` when `status ∈ {acquired, public}` **or** `top_company is True`. This matches
`train.py` exactly (the `y_fund` definition).

In [ ]:
status = df["status"].astype(str).str.strip().str.lower()
top = df.get("top_company")
top_bool = (top.astype(str).str.lower().eq("true").to_numpy()
            if top is not None else np.zeros(len(df), bool))
y = (status.isin(["acquired", "public"]).to_numpy() | top_bool).astype(int)

base_rate = y.mean()
print(f"positives (fundable/success): {int(y.sum()):,}  /  {len(y):,}")
print(f"base success rate: {base_rate:.1%}")
print("\npositive composition:")
print(pd.Series(np.where(top_bool & (y == 1), "top_company",
      np.where(status.isin(['acquired','public']) & (y==1), 'acquired/public', '')))
      .replace('', np.nan).dropna().value_counts())

## 3 · Feature engineering — the 6 founder-suppliable features

These are the **only** signals a founder can give in the onboarding questionnaire, so the
model can only learn from them. Code mirrors `app/ml_training/features.py` line for line —
keep the two in sync if either changes.

| Feature | Meaning |
|---|---|
| `team_size` | employee count |
| `company_age_years` | years since launch |
| `industry_code` | ordinal industry id (target-encoded inside the pipeline) |
| `stage_code` | ordinal funding-stage ladder (0=Idea … 8=Public, -1 unknown) |
| `age_stage_gap` | years older than typical for the declared stage (stagnation signal) |
| `team_per_year` | hiring-velocity proxy (team size per year of life) |

In [ ]:
import datetime as _dt

FEATURE_NAMES = ["team_size", "company_age_years", "industry_code",
                 "stage_code", "age_stage_gap", "team_per_year"]
INDUSTRY_COL_INDEX = FEATURE_NAMES.index("industry_code")

_STAGE_ORDINAL = {"idea":0, "pre-seed":1, "preseed":1, "seed":2, "early":3,
                  "series a":4, "series b":5, "growth":6, "series c":7,
                  "series d":7, "late":7, "public":8}
_EXPECTED_AGE_AT_STAGE = {0:0.0, 1:0.5, 2:1.0, 3:2.0, 4:3.0, 5:5.0, 6:6.0, 7:8.0, 8:10.0}

def stage_to_code(stage):
    if not stage or str(stage) == "nan":
        return -1
    return _STAGE_ORDINAL.get(str(stage).strip().lower(), -1)

def _age_from_unix(ts):
    try:
        launched = _dt.datetime.utcfromtimestamp(float(ts))
        return max(0.0, round((_dt.datetime.utcnow() - launched).days / 365.25, 2))
    except Exception:
        return 0.0

def _derive(team, age, ind_code, stage_code):
    expected = _EXPECTED_AGE_AT_STAGE.get(stage_code, 3.0)
    return [team, age, float(ind_code), float(stage_code),
            round(age - expected, 2), round(team / (age + 1.0), 2)]

def row_to_features(row, industry_map):
    team = row.get("team_size")
    team = float(team) if team is not None and str(team) != "nan" else 0.0
    age = _age_from_unix(row.get("launched_at"))
    ind_code = industry_map.get(str(row.get("industry", "")).strip(), -1)
    return _derive(team, age, ind_code, stage_to_code(row.get("stage")))

# Leakage-safe: industry_map is built from the full corpus (a lookup table, not a fit).
industry_map = {name: i for i, name in enumerate(
    sorted(str(x).strip() for x in df["industry"].dropna().unique()))}
X = np.array([row_to_features(r, industry_map) for r in df.to_dict(orient="records")],
             dtype=float)
print("X shape:", X.shape, "| industries:", len(industry_map))
pd.DataFrame(X, columns=FEATURE_NAMES).describe().round(2)

## 4 · Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

# Success rate vs team size (bucketed)
team = X[:, FEATURE_NAMES.index("team_size")]
bins = [0, 1, 5, 10, 25, 50, 100, 1e9]
labels = ["0", "1-5", "6-10", "11-25", "26-50", "51-100", "100+"]
tb = pd.cut(team, bins=bins, labels=labels, include_lowest=True)
pd.Series(y).groupby(tb).mean().plot(kind="bar", ax=ax[0], color="#4f9")
ax[0].axhline(base_rate, ls="--", c="k", alpha=.5, label="base rate")
ax[0].set(title="Success rate by team size", ylabel="P(success)", xlabel="team size")
ax[0].legend()

# Success rate vs company age (bucketed)
age = X[:, FEATURE_NAMES.index("company_age_years")]
ab = pd.cut(age, bins=[-.1, 1, 3, 5, 8, 12, 1e9],
            labels=["<1", "1-3", "3-5", "5-8", "8-12", "12+"])
pd.Series(y).groupby(ab).mean().plot(kind="bar", ax=ax[1], color="#9af")
ax[1].axhline(base_rate, ls="--", c="k", alpha=.5, label="base rate")
ax[1].set(title="Success rate by company age", ylabel="P(success)", xlabel="years")
ax[1].legend()
plt.tight_layout(); plt.show()

## 5 · Stratified train / test split

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, random_state=RNG, stratify=y)
print(f"train {len(Xtr):,} · test {len(Xte):,} · test success rate {yte.mean():.1%}")

## 6 · Pipeline + hyperparameter search

`TargetEncoder` is **cross-fitted** inside the pipeline, so the industry encoding never
sees its own row's label — no target leakage. The classifier is a heavily regularized
`HistGradientBoosting` (only 6 features, so we keep trees small).

In [ ]:
def make_pipeline():
    encoder = ColumnTransformer(
        [("industry_te", TargetEncoder(target_type="binary", random_state=RNG),
          [INDUSTRY_COL_INDEX])],
        remainder="passthrough")
    clf = HistGradientBoostingClassifier(max_iter=400, random_state=RNG)
    return Pipeline([("encode", encoder), ("clf", clf)])

PARAM_GRID = {
    "clf__learning_rate": [0.03, 0.05],
    "clf__max_leaf_nodes": [7, 15],
    "clf__min_samples_leaf": [40, 80],
    "clf__l2_regularization": [0.0, 1.0],
    "clf__class_weight": [None, "balanced"],
}
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG)

search = GridSearchCV(make_pipeline(), PARAM_GRID, cv=CV, scoring="roc_auc", n_jobs=-1)
search.fit(Xtr, ytr)
cv_auc_mean = float(search.cv_results_["mean_test_score"][search.best_index_])
cv_auc_std  = float(search.cv_results_["std_test_score"][search.best_index_])
print(f"best CV ROC-AUC = {cv_auc_mean:.3f} ± {cv_auc_std:.3f}")
print("best params:", {k.replace('clf__',''): v for k,v in search.best_params_.items()})

## 7 · Isotonic calibration

Grid search optimizes *ranking* (AUC); calibration makes the output a *trustworthy
probability* you can read as a real funding likelihood.

In [ ]:
model = CalibratedClassifierCV(search.best_estimator_, method="isotonic", cv=CV)
model.fit(Xtr, ytr)

p_raw = search.best_estimator_.predict_proba(Xte)[:, 1]
p_cal = model.predict_proba(Xte)[:, 1]
print(f"Brier  raw={brier_score_loss(yte, p_raw):.4f}  calibrated={brier_score_loss(yte, p_cal):.4f}")

fig, ax = plt.subplots(figsize=(6, 5))
for label, p in [("raw", p_raw), ("calibrated", p_cal)]:
    frac, mean = calibration_curve(yte, p, n_bins=10, strategy="quantile")
    ax.plot(mean, frac, "o-", label=label)
ax.plot([0, 1], [0, 1], "k--", alpha=.4, label="perfect")
ax.set(xlabel="predicted probability", ylabel="observed success rate", title="Calibration")
ax.legend(); plt.show()

## 8 · Honest holdout evaluation

In [ ]:
auc    = roc_auc_score(yte, p_cal)
pr_auc = average_precision_score(yte, p_cal)
brier  = brier_score_loss(yte, p_cal)
acc    = accuracy_score(yte, (p_cal >= 0.5).astype(int))

best_t, best_f1 = 0.5, 0.0
for t in np.arange(0.1, 0.9, 0.02):
    f1 = f1_score(yte, (p_cal >= t).astype(int), zero_division=0)
    if f1 > best_f1:
        best_t, best_f1 = float(t), float(f1)

print(f"ROC-AUC = {auc:.3f}   PR-AUC = {pr_auc:.3f}   Brier = {brier:.3f}   acc = {acc:.3f}")
print(f"best F1 = {best_f1:.3f} at threshold {best_t:.2f}")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
fpr, tpr, _ = roc_curve(yte, p_cal)
ax[0].plot(fpr, tpr, label=f"AUC={auc:.3f}"); ax[0].plot([0,1],[0,1],"k--",alpha=.4)
ax[0].set(title="ROC", xlabel="FPR", ylabel="TPR"); ax[0].legend()
prec, rec, _ = precision_recall_curve(yte, p_cal)
ax[1].plot(rec, prec, label=f"PR-AUC={pr_auc:.3f}"); ax[1].axhline(yte.mean(), ls="--", c="k", alpha=.4)
ax[1].set(title="Precision-Recall", xlabel="recall", ylabel="precision"); ax[1].legend()
ConfusionMatrixDisplay(confusion_matrix(yte, (p_cal >= best_t).astype(int)),
                       display_labels=["not funded", "funded"]).plot(ax=ax[2], colorbar=False)
ax[2].set(title=f"Confusion @ {best_t:.2f}")
plt.tight_layout(); plt.show()

## 9 · What drives the funding score

Calibrated pipelines hide native tree importances, so we use **model-agnostic permutation
importance** (drop in ROC-AUC when each feature is shuffled) — exactly what `train.py`
stores in the bundle.

In [ ]:
perm = permutation_importance(model, Xte, yte, scoring="roc_auc",
                              n_repeats=10, random_state=RNG, n_jobs=-1)
imp = (pd.Series(perm.importances_mean, index=FEATURE_NAMES)
       .sort_values())
ax = imp.plot(kind="barh", figsize=(7, 3.5), color="#4f9")
ax.set(title="Permutation importance (Δ ROC-AUC)", xlabel="importance")
plt.tight_layout(); plt.show()
importances = {n: round(float(v), 4) for n, v in zip(FEATURE_NAMES, perm.importances_mean)}
importances

## 10 · Inference contract — base probability + traction adjustment

At runtime `app/agents/ml/funding_readiness.py` takes the model's **base probability**, then
nudges it with the founder's *own* traction numbers (runway, growth, revenue) using weights
from `config.json → funding_adjustment`, clamped to `[0.02, 0.98]`. The model alone never sees
those financials — it only sees the 6 structural features. We reproduce that here.

In [ ]:
def metrics_to_features(m, industry_map):
    """Questionnaire metrics -> the same 6-feature vector (mirrors features.py)."""
    team = float(m.get("employee_count", 0) or 0)
    age = m.get("company_age_years")
    if age is None:
        yr = m.get("founding_year")
        age = max(0, _dt.datetime.utcnow().year - int(yr)) if yr else 0
    ind_code = industry_map.get(str(m.get("industry", "")).strip(), -1)
    return _derive(float(team), float(age), float(ind_code), stage_to_code(m.get("stage")))

def predict_funding(m):
    x = np.array([metrics_to_features(m, industry_map)])
    base = float(model.predict_proba(x)[0][1])
    # Illustrative traction layer (real weights live in config.json -> funding_adjustment).
    runway, growth, revenue = m.get("runway",0), m.get("customer_growth",0), m.get("revenue",0)
    d = (min(max(growth,0)/100, 1) * 0.10
         + min(runway/18, 1) * 0.08
         + min(revenue/1_000_000, 1) * 0.07
         - (0.10 if runway < 3 else 0.0))
    prob = max(0.02, min(0.98, base + d))
    return {"funding_probability": round(prob,3), "investor_score": round(prob*100,1),
            "model_base_probability": round(base,3)}

examples = [
    {"employee_count":18, "founding_year":2022, "industry":"B2B", "stage":"Series A",
     "runway":14, "customer_growth":40, "revenue":600_000},
    {"employee_count":2,  "founding_year":2017, "industry":"Consumer", "stage":"Seed",
     "runway":2, "customer_growth":3, "revenue":0},
]
for e in examples:
    r = predict_funding(e)
    print(f"{e['industry']:9} {e['stage']:9}  base={r['model_base_probability']:.2f}  "
          f"-> investor_score={r['investor_score']:5.1f}/100")

## 11 · Persist the bundle → `models/funding_model.joblib`

Saved with the **exact schema** the app loader expects (`app/ml_training/loader.py`), so this
is a drop-in replacement for the trainer's output. The keys match `train.py` so
`funding_readiness.py` reads `model`, `industry_map`, `model_type`, `n_samples`, and
`metrics['auc']` unchanged.

In [ ]:
bundle = {
    "model": model,
    "feature_names": FEATURE_NAMES,
    "industry_map": industry_map,
    "feature_importances": importances,
    "best_params": {k.replace("clf__",""): v for k,v in search.best_params_.items()},
    "metrics": {
        "auc": round(float(auc), 4),
        "accuracy": round(float(acc), 4),
        "pr_auc": round(float(pr_auc), 4),
        "brier": round(float(brier), 4),
        "cv_auc_mean": round(cv_auc_mean, 4),
        "cv_auc_std": round(cv_auc_std, 4),
        "best_f1": round(best_f1, 4),
        "best_f1_threshold": round(best_t, 2),
    },
    "n_samples": int(len(X)),
    "positives": int(y.sum()),
    "model_type": "HistGradientBoosting (calibrated)",
}
out = MODELS / "funding_model.joblib"
joblib.dump(bundle, out)
print("saved ->", out)
print("metrics:", bundle["metrics"])